In [44]:
#==================================
# LandCover 分类占比提取脚本
# 作者: King (Gemini 优化版)
# 日期: 2026-02-20
#==================================

import arcpy
import os
import pandas as pd
import json
import time
import sys

# ==================== 参数配置 ====================
# 1. Landcover 栅格路径
lc_raster = r"C:\Users\King\Documents\ArcGIS\Projects\MyProject8\MyProject8.gdb\Landfire_5Km"

# 2. 5km 缓冲区图层路径
buffer_layer = r"C:\Users\King\Documents\ArcGIS\Projects\MyProject8\MyProject8.gdb\Buffer_5km_Test"

# 3. 已经解析好的 JSON 映射文件路径
mapping_json = r"C:\Users\King\Desktop\大三下学习生活\prj-Alban\data\raw\landcover\lc_mapping.json"

# 4. 最终结果输出路径
output_csv = r"C:\Users\King\Desktop\大三下学习生活\prj-Alban\data\output\LandCover_Stats_Final.csv"

# ==================== 环境初始化 ====================
arcpy.env.overwriteOutput = True
if arcpy.CheckExtension("Spatial") == "Available":
    arcpy.CheckOutExtension("Spatial")
else:
    print("错误：无法获取 Spatial Analyst 许可。")
    sys.exit()

# 直接加载已有的 JSON 字典
print(f"正在读取映射文件: {os.path.basename(mapping_json)}")
with open(mapping_json, 'r', encoding='utf-8') as f:
    lc_map = json.load(f)

# ==================== 执行核心统计 ====================
desc = arcpy.Describe(buffer_layer)
oid_fieldname = desc.OIDFieldName
total_points = int(arcpy.management.GetCount(buffer_layer)[0])

all_results = []
start_time = time.time()
processed_count = 0
consecutive_errors = 0

print(f"--- 任务启动：共计 {total_points} 个点 ---")

with arcpy.da.SearchCursor(buffer_layer, ["SHAPE@", oid_fieldname]) as cursor:
    for row in cursor:
        buffer_geom = row[0]
        oid_val = row[1]
        
        try:
            # 动态限制分析范围以提速 [cite: 2026-02-19]
            arcpy.env.extent = buffer_geom.extent
            temp_table = r"memory\lc_tab_temp"
            
            # 使用 Tabulate Area 统计区域内各 Value 的面积占比
            arcpy.sa.TabulateArea(buffer_layer, oid_fieldname, lc_raster, "Value", temp_table)
            
            # 获取输出表的字段 (VALUE_X)
            fields = [f.name for f in arcpy.ListFields(temp_table) if f.name.startswith("VALUE_")]
            
            with arcpy.da.SearchCursor(temp_table, fields) as t_cursor:
                for t_row in t_cursor:
                    total_area = sum(t_row)
                    if total_area > 0:
                        record = {"OID": oid_val}
                        for i, f_name in enumerate(fields):
                            # 提取 Value 代码 (例如从 VALUE_1 提取 1)
                            val_code = f_name.split("_")[-1]
                            
                            # 从 JSON 中获取名称，若无则使用原始代码，并清理空格为下划线
                            raw_name = lc_map.get(str(val_code), f"Code_{val_code}")
                            clean_name = "".join(c if c.isalnum() else "_" for c in raw_name)
                            
                            # 计算占比
                            record[clean_name] = round(t_row[i] / total_area, 4)
                        all_results.append(record)

            processed_count += 1
            consecutive_errors = 0 # 成功则重置错误计数
            
            # --- 进度监控与 ETA 计算 ---
            if processed_count % 5 == 0 or processed_count == total_points:
                elapsed = time.time() - start_time
                speed = processed_count / elapsed # 点/秒
                percent = (processed_count / total_points) * 100
                remaining = total_points - processed_count
                eta_min = (remaining / speed / 60) if speed > 0 else 0
                
                print(f"进度: {percent:.1f}% ({processed_count}/{total_points}) | "
                      f"速度: {speed:.2f} 点/秒 | 预计剩余: {eta_min:.1f} 分钟")

            arcpy.management.Delete(temp_table)

        except Exception as e:
            consecutive_errors += 1
            print(f"!!! OID {oid_val} 处理失败: {e} (连续失败: {consecutive_errors})")
            
            if consecutive_errors >= 3:
                print("\n[熔断] 连续 3 次提取失败。请检查栅格数据路径或空间重叠。")
                break

# ==================== 结果保存 ====================
if all_results:
    print(f"正在整合数据并导出 CSV...")
    df = pd.DataFrame(all_results).fillna(0)
    # 保证 OID 在第一列
    df = df[["OID"] + [c for c in df.columns if c != "OID"]]
    df.to_csv(output_csv, index=False)
    
    total_time = (time.time() - start_time) / 60
    print(f"--- 任务完成 --- \n成功提取: {len(df)} \n总耗时: {total_time:.2f} 分钟")
else:
    print("错误：未采集到任何有效数据。")

arcpy.CheckInExtension("Spatial")

正在读取映射文件: lc_mapping.json
--- 任务启动：共计 4 个点 ---
进度: 100.0% (4/4) | 速度: 0.87 点/秒 | 预计剩余: 0.0 分钟
正在整合数据并导出 CSV...
--- 任务完成 --- 
成功提取: 16 
总耗时: 0.08 分钟


'CheckedIn'

In [ ]:
# ==================== 执行核心统计 ====================
# 关键修复：清除所有选择状态，确保获取的是图层完整要素总数 [cite: 2026-02-19, 2026-02-20]
arcpy.management.SelectLayerByAttribute(buffer_layer, "CLEAR_SELECTION")

# 获取准确的要素总数
total_points = int(arcpy.management.GetCount(buffer_layer)[0])
desc = arcpy.Describe(buffer_layer)
oid_fieldname = desc.OIDFieldName

all_results = []
start_time = time.time()
processed_count = 0
consecutive_errors = 0

print(f"--- 任务启动：共计 {total_points} 个点 ---")

# 使用 SearchCursor 遍历，它会自动处理即使有数万个点的性能问题 [cite: 2026-02-19, 2026-02-20]
with arcpy.da.SearchCursor(buffer_layer, ["SHAPE@", oid_fieldname]) as cursor:
    for row in cursor:
        buffer_geom = row[0]
        oid_val = row[1]
        
        try:
            # 动态限制分析范围提速 [cite: 2026-02-19]
            arcpy.env.extent = buffer_geom.extent
            temp_table = r"memory\lc_tab_temp"
            if arcpy.Exists(temp_table):
                arcpy.management.Delete(temp_table)
            
            # 统计
            arcpy.sa.TabulateArea(buffer_layer, oid_fieldname, lc_raster, "Value", temp_table)
            
            fields = [f.name for f in arcpy.ListFields(temp_table) if f.name.startswith("VALUE_")]
            
            with arcpy.da.SearchCursor(temp_table, fields) as t_cursor:
                for t_row in t_cursor:
                    total_area = sum(t_row)
                    if total_area > 0:
                        record = {"OID": oid_val}
                        for i, f_name in enumerate(fields):
                            val_code = f_name.split("_")[-1]
                            raw_name = lc_map.get(str(val_code), f"Code_{val_code}")
                            clean_name = "".join(c if c.isalnum() else "_" for c in raw_name)
                            record[clean_name] = round(t_row[i] / total_area, 4)
                        all_results.append(record)

            # 成功处理计数
            processed_count += 1
            consecutive_errors = 0 
            
            # --- 进度监控与 ETA 计算 ---
            if processed_count % 5 == 0 or processed_count == total_points:
                elapsed = time.time() - start_time
                speed = processed_count / elapsed 
                percent = (processed_count / total_points) * 100
                remaining = total_points - processed_count
                eta_min = (remaining / speed / 60) if speed > 0 else 0
                
                print(f"进度: {percent:.1f}% ({processed_count}/{total_points}) | "
                      f"速度: {speed:.2f} 点/秒 | 预计剩余时间: {eta_min:.1f} 分钟")
            if percent >= 100:
                print("进度大于100%,bug了")
                break

            arcpy.management.Delete(temp_table)

        except Exception as e:
            consecutive_errors += 1
            print(f"!!! OID {oid_val} 处理失败: {e} (连续失败: {consecutive_errors})")
            
            if consecutive_errors >= 3:
                print("\n[熔断] 连续 3 次提取失败。请检查栅格数据路径或空间参考是否统一。")
                break

--- 任务启动：共计 202 个点 ---
!!! OID 1 处理失败: name 'lc_map' is not defined (连续失败: 1)
!!! OID 2 处理失败: 执行失败。参数无效。
ERROR 000872: 输出表: 数据集 memory\lc_tab_temp 已存在，由于覆盖现有数据集选项已禁用，因此无法覆盖该数据集。
执行(TabulateArea)失败。
 (连续失败: 2)
!!! OID 3 处理失败: 执行失败。参数无效。
ERROR 000872: 输出表: 数据集 memory\lc_tab_temp 已存在，由于覆盖现有数据集选项已禁用，因此无法覆盖该数据集。
执行(TabulateArea)失败。
 (连续失败: 3)

[熔断] 连续 3 次提取失败。请检查栅格数据路径或空间参考是否统一。


In [1]:
import arcpy
import os
import pandas as pd
import json
import time
import sys

# ==================== 参数配置 ====================
# 1. Landcover 栅格路径
lc_raster = r"C:\Users\King\Documents\ArcGIS\Projects\MyProject8\MyProject8.gdb\Landfire_5Km"

# 2. 5km 缓冲区图层路径
buffer_layer = r"C:\Users\King\Documents\ArcGIS\Projects\MyProject8\MyProject8.gdb\Buffer_5km"

# 3. 已经解析好的 JSON 映射文件路径
mapping_json = r"C:\Users\King\Desktop\大三下学习生活\prj-Alban\data\raw\landcover\lc_mapping.json"

# 4. 最终结果输出路径
output_csv = r"C:\Users\King\Desktop\大三下学习生活\prj-Alban\data\output\LandCover_Stats_Final.csv"
arcpy.management.SelectLayerByAttribute(buffer_layer, "CLEAR_SELECTION")

# 获取准确的要素总数
total_points = int(arcpy.management.GetCount(buffer_layer)[0])
desc = arcpy.Describe(buffer_layer)
oid_fieldname = desc.OIDFieldName

print(f"点总数为: {total_points}, OID 字段名为: {oid_fieldname}")

点总数为: 202, OID 字段名为: OBJECTID
